# Лагерные хроники (оконная версия)
Полноценный запуск в окне на tkinter: кнопки, сюжетный лог, параметры героя и сохранение итогов.

In [ ]:
import tkinter as tk


class Character:
    def __init__(self, name, role):
        self.name = name
        self.role = role


class PlayerState:
    def __init__(self, name):
        self.name = name
        self.energy = 3
        self.reputation = 0
        self.courage = 0
        self.inventory = []
        self.clues = set()
        self.route = []
        self.ending = ''


class GameEngine:
    def __init__(self, player_name):
        self.player = PlayerState(player_name)
        self.characters = [
            Character('Алиса', 'лидер отряда'),
            Character('Электроник', 'техник'),
            Character('Ольга Дмитриевна', 'вожатая')
        ]
        self.locations = {
            'площадь': 'У памятника свежие следы на влажной земле.',
            'сцена': 'За кулисами лежит список выступлений с пометками.',
            'столовая': 'Дежурные шепчутся о ночном силуэте у склада.',
            'лодочная': 'Под досками тайник с ржавым замком.',
            'радиорубка': 'В журнале дежурств не хватает страницы.'
        }
        self.suspicions = {'Алиса': 1, 'Электроник': 2, 'Ольга Дмитриевна': 0}
        self.inspect_order = list(self.locations.keys())
        self.inspect_index = 0
        self.checked_places = 0

    def begin(self):
        self.player.route.append('Прибытие в лагерь')
        return 'Лагерь "Солнечная Заря". Ночью пропал архивный ящик с документами. Нужно провести расследование.'

    def apply_day_one(self, choice):
        if choice == 'help':
            self.player.reputation += 2
            self.player.inventory.append('блокнот вожатой')
            self.player.route.append('Помощь вожатой')
            return 'Ты помог(ла) вожатой собрать показания. Репутация выросла.'
        self.player.courage += 2
        self.player.energy -= 1
        self.player.inventory.append('старый компас')
        self.player.route.append('Одиночный поиск')
        return 'Ты пошел(ла) по следу один(одна). Смелость выросла, но сил стало меньше.'

    def has_next_location(self):
        return self.inspect_index < len(self.inspect_order) and self.checked_places < 3

    def next_location(self):
        if not self.has_next_location():
            return None
        return self.inspect_order[self.inspect_index]

    def inspect_location(self, inspect_yes):
        place = self.inspect_order[self.inspect_index]
        self.inspect_index += 1
        if not inspect_yes:
            return 'Ты пропустил(а) локацию: ' + place

        self.checked_places += 1
        self.player.route.append('Осмотр: ' + place)
        self.player.clues.add(place)
        self.player.inventory.append('улика: ' + place)

        if place == 'лодочная':
            self.player.courage += 1
        if place == 'столовая':
            self.player.reputation += 1

        return self.locations[place]

    def radio_attempt(self, code):
        if code == '1989':
            self.player.clues.add('код-1989')
            self.player.reputation += 1
            self.player.route.append('Радиорубка взломана')
            return True, 'Код верный. Найдено имя дежурного из журнала.'
        return False, 'Код неверный.'

    def night_action(self, action):
        if action == 'chase':
            self.player.courage += 1
            self.player.energy -= 1
            self.player.route.append('Ночное преследование')
            self.player.clues.add('следы у склада')
            return 'Ты преследовал(а) силуэт и нашел(ла) новые следы.'
        self.player.reputation += 1
        self.player.route.append('Позвал помощь')
        return 'Ты позвал(а) помощь и усилил(а) доверие отряда.'

    def optimize_inventory(self):
        if len(self.player.inventory) > 6:
            self.player.inventory.pop(0)
        if 'старый компас' in self.player.inventory and 'блокнот вожатой' in self.player.inventory:
            self.player.inventory.remove('старый компас')
            self.player.inventory.append('карта маршрутов')
        if 'улика: сцена' in self.player.inventory:
            self.player.clues.add('переписанный сценарий')

    def final_decision(self, accuse):
        if accuse == 'Алиса':
            self.suspicions['Алиса'] += 2
            self.player.route.append('Обвинение Алисы')
        elif accuse == 'Электроник':
            self.suspicions['Электроник'] += 2
            self.player.route.append('Обвинение Электроника')
        else:
            self.player.reputation += 1
            self.player.route.append('Никого не обвинял, анализировал факты')

        many_clues = len(self.player.clues) >= 3
        trusted = self.player.reputation >= 3
        brave = self.player.courage >= 2

        if many_clues and trusted and brave and accuse == 'Никого':
            self.player.ending = 'Истинная концовка: ты раскрываешь подмену архивов и спасаешь смену.'
        elif many_clues and accuse in ('Алиса', 'Электроник'):
            self.player.ending = 'Драматичная концовка: виновный назван, но лагерь расколот.'
        elif trusted or many_clues:
            self.player.ending = 'Нейтральная концовка: часть правды открыта, но не вся.'
        else:
            self.player.ending = 'Плохая концовка: дело закрыли без убедительных доказательств.'

        return self.player.ending

    def build_summary(self):
        summary = {}
        summary['герой'] = self.player.name
        summary['энергия'] = self.player.energy
        summary['репутация'] = self.player.reputation
        summary['смелость'] = self.player.courage
        summary['маршрут'] = self.player.route
        summary['улики'] = list(self.player.clues)
        summary['инвентарь'] = self.player.inventory
        summary['подозрения'] = self.suspicions
        summary['финал'] = self.player.ending
        return summary

    def save_summary(self, path='novel_result.txt'):
        s = self.build_summary()
        lines = [
            'Герой: ' + s['герой'],
            'Энергия: ' + str(s['энергия']),
            'Репутация: ' + str(s['репутация']),
            'Смелость: ' + str(s['смелость']),
            'Маршрут:'
        ]
        for step in s['маршрут']:
            lines.append('- ' + step)
        lines.append('Улики: ' + ', '.join(s['улики']))
        lines.append('Инвентарь: ' + ', '.join(s['инвентарь']))
        lines.append('Подозрения: ' + str(s['подозрения']))
        lines.append('Концовка: ' + s['финал'])
        with open(path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))


class NovelWindow:
    def __init__(self, root):
        self.root = root
        self.root.title('Лагерные хроники')
        self.root.geometry('900x620')
        self.engine = None
        self.radio_attempts = 3

        self.title_label = tk.Label(root, text='Лагерные хроники', font=('Arial', 18, 'bold'))
        self.title_label.pack(pady=8)

        self.story = tk.Text(root, wrap='word', height=20, width=110, state='disabled')
        self.story.pack(padx=10, pady=8)

        self.stats_label = tk.Label(root, text='')
        self.stats_label.pack(pady=6)

        self.controls = tk.Frame(root)
        self.controls.pack(pady=8)

        self.name_entry = tk.Entry(self.controls, width=30)
        self.name_entry.grid(row=0, column=0, padx=4)
        self.name_entry.insert(0, 'Семён')

        self.start_button = tk.Button(self.controls, text='Начать игру', command=self.start_game)
        self.start_button.grid(row=0, column=1, padx=4)

        self.action_frame = tk.Frame(root)
        self.action_frame.pack(pady=10)

    def clear_actions(self):
        for w in self.action_frame.winfo_children():
            w.destroy()

    def log(self, text):
        self.story.configure(state='normal')
        self.story.insert('end', text + '\\n\\n')
        self.story.see('end')
        self.story.configure(state='disabled')

    def update_stats(self):
        p = self.engine.player
        self.stats_label.config(
            text='Герой: {0} | Энергия: {1} | Репутация: {2} | Смелость: {3} | Улик: {4}'.format(
                p.name, p.energy, p.reputation, p.courage, len(p.clues)
            )
        )

    def start_game(self):
        name = self.name_entry.get().strip() or 'Семён'
        self.engine = GameEngine(name)
        self.story.configure(state='normal')
        self.story.delete('1.0', 'end')
        self.story.configure(state='disabled')
        self.log(self.engine.begin())
        self.update_stats()
        self.show_day_one()

    def show_day_one(self):
        self.clear_actions()
        tk.Button(
            self.action_frame,
            text='Помочь вожатой собрать показания',
            command=lambda: self.day_one_choice('help')
        ).pack(fill='x', pady=3)
        tk.Button(
            self.action_frame,
            text='Пойти по следу в одиночку',
            command=lambda: self.day_one_choice('solo')
        ).pack(fill='x', pady=3)

    def day_one_choice(self, choice):
        self.log(self.engine.apply_day_one(choice))
        self.update_stats()
        self.show_inspection()

    def show_inspection(self):
        self.clear_actions()
        if not self.engine.has_next_location():
            self.log('Осмотр локаций завершен.')
            self.show_radio_stage()
            return

        place = self.engine.next_location()
        tk.Label(self.action_frame, text='Осмотреть локацию: ' + place + '?').pack(pady=4)
        tk.Button(
            self.action_frame,
            text='Осмотреть',
            command=lambda: self.inspect_choice(True)
        ).pack(fill='x', pady=3)
        tk.Button(
            self.action_frame,
            text='Пропустить',
            command=lambda: self.inspect_choice(False)
        ).pack(fill='x', pady=3)

    def inspect_choice(self, inspect_yes):
        self.log(self.engine.inspect_location(inspect_yes))
        self.update_stats()
        self.show_inspection()

    def show_radio_stage(self):
        self.clear_actions()
        tk.Label(self.action_frame, text='Радиорубка: введи 4-значный код (3 попытки)').pack(pady=4)
        self.code_entry = tk.Entry(self.action_frame, width=12)
        self.code_entry.pack(pady=3)
        tk.Button(self.action_frame, text='Проверить код', command=self.submit_code).pack(fill='x', pady=3)

    def submit_code(self):
        code = self.code_entry.get().strip()
        ok, msg = self.engine.radio_attempt(code)
        self.log(msg)
        if ok:
            self.update_stats()
            self.show_night_stage()
            return

        self.radio_attempts -= 1
        if self.radio_attempts <= 0:
            self.log('Попытки закончились. Радиорубку открыть не удалось.')
            self.engine.player.route.append('Радиорубка не взломана')
            self.show_night_stage()
        else:
            self.log('Осталось попыток: ' + str(self.radio_attempts))

    def show_night_stage(self):
        self.clear_actions()
        tk.Label(self.action_frame, text='Ночной эпизод у склада').pack(pady=4)
        tk.Button(
            self.action_frame,
            text='Преследовать силуэт',
            command=lambda: self.night_choice('chase')
        ).pack(fill='x', pady=3)
        tk.Button(
            self.action_frame,
            text='Позвать помощь',
            command=lambda: self.night_choice('help')
        ).pack(fill='x', pady=3)

    def night_choice(self, action):
        self.log(self.engine.night_action(action))
        self.engine.optimize_inventory()
        self.update_stats()
        self.show_final_stage()

    def show_final_stage(self):
        self.clear_actions()
        tk.Label(self.action_frame, text='Кого обвинить на общем сборе?').pack(pady=4)
        tk.Button(
            self.action_frame,
            text='Обвинить Алису',
            command=lambda: self.finish_game('Алиса')
        ).pack(fill='x', pady=3)
        tk.Button(
            self.action_frame,
            text='Обвинить Электроника',
            command=lambda: self.finish_game('Электроник')
        ).pack(fill='x', pady=3)
        tk.Button(
            self.action_frame,
            text='Никого не обвинять, опереться на факты',
            command=lambda: self.finish_game('Никого')
        ).pack(fill='x', pady=3)

    def finish_game(self, accuse):
        ending = self.engine.final_decision(accuse)
        self.engine.save_summary('novel_result.txt')
        self.update_stats()
        self.clear_actions()
        self.log('Финал: ' + ending)
        self.log('Итог сохранен в novel_result.txt')
        tk.Button(self.action_frame, text='Начать заново', command=self.start_game).pack(fill='x', pady=3)


root = tk.Tk()
app = NovelWindow(root)
root.mainloop()
